
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 2L - Introduction to Spark Structured Streaming

In this lab, you'll work with a streaming dataset containing order status updates. You'll learn how to create streaming DataFrames, perform basic transformations, and work with different streaming sinks.

### Objectives
- Understand stream processing fundamentals
- Implement basic streaming operations
- Work with different streaming sources and sinks
- Apply streaming transformations and watermarking
- Handle late data and monitor streaming queries

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Stream Processing Setup

First, let's set up our streaming infrastructure and define our data schema.

In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *
# Define the schema
schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("order_status", StringType(), True),
    StructField("status_timestamp", LongType(), True)
])

# Create a streaming DataFrame using this schema
status_stream = spark.readStream \
    .format("json") \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/status/stream_json") \
    .load()

# Verify it's a streaming DataFrame
print(f"isStreaming: {status_stream.isStreaming}")

isStreaming: True


## B. Streaming Queries

Now we will create a basic streaming query, using the memory sink which we will subsequently query using SQL.

In [0]:
%python

# Write the results of your status_stream into a memory sink with a query name of "order_status_streaming_table", appending records to the output sink
# Stop any existing queries with the same name

for q in spark.streams.active:
    if q.name == "order_status_streaming_table":
        q.stop()

# Write to memory sink for interactive querying
memory_query = status_stream.writeStream \
    .format("memory") \
    .queryName("order_status_streaming_table") \
    .outputMode("append") \
    .start()

In [0]:
SELECT order_status, count(*) as cnt 
FROM order_status_streaming_table
GROUP BY order_status

order_status,cnt
on the way,769
canceled,85
return canceled,22
reported shipping error,39
delivered,740
return processed,109
return picked up,117
placed,917
preparing,807
return requested,137


## C. Basic Transformations

Now, you'll perform some basic transformations on the streaming data.

In [0]:
%python
# Perform basic transformations
transformed_stream = status_stream \
    .withColumn("event_time", from_unixtime(col("status_timestamp") / 1000).cast("timestamp")) \
    .withColumn("status_description", concat(lit("Order status: "), col("order_status"))) \
    .withColumn("is_completed", col("order_status").isin("delivered", "canceled"))

# Display the transformed stream
display(transformed_stream)

order_id,order_status,status_timestamp,event_time,status_description,is_completed
75123,placed,1640392092,1970-01-19T23:39:52Z,Order status: placed,false
75124,placed,1640392500,1970-01-19T23:39:52Z,Order status: placed,false
75125,placed,1640394862,1970-01-19T23:39:54Z,Order status: placed,false
75126,placed,1640396067,1970-01-19T23:39:56Z,Order status: placed,false
75127,placed,1640399066,1970-01-19T23:39:59Z,Order status: placed,false
75128,placed,1640404853,1970-01-19T23:40:04Z,Order status: placed,false
75129,placed,1640407272,1970-01-19T23:40:07Z,Order status: placed,false
75130,placed,1640419989,1970-01-19T23:40:19Z,Order status: placed,false
75131,placed,1640422131,1970-01-19T23:40:22Z,Order status: placed,false
75132,placed,1640423697,1970-01-19T23:40:23Z,Order status: placed,false


## D. Controlling Processing with Triggers

Finally, you'll use triggers to control how the stream processes data.

In [0]:
%python

# Stop any existing queries with the same name
for q in spark.streams.active:
    if q.name == "triggered_status_updates":
        q.stop()
        
# Create a triggered streaming query
triggered_query = status_stream \
    .withColumn("processing_time", current_timestamp()) \
    .writeStream \
    .format("memory") \
    .queryName("triggered_status_updates") \
    .outputMode("append") \
    .trigger(processingTime="15 seconds") \
    .start()

In [0]:

-- Check the processing batches
SELECT 
  processing_time,
  order_status,
  count(*) as record_count
FROM triggered_status_updates
GROUP BY processing_time, order_status
ORDER BY processing_time, order_status

processing_time,order_status,record_count
2025-07-01T21:50:07.481Z,canceled,8
2025-07-01T21:50:07.481Z,delivered,94
2025-07-01T21:50:07.481Z,on the way,107
2025-07-01T21:50:07.481Z,placed,174
2025-07-01T21:50:07.481Z,preparing,109
2025-07-01T21:50:07.481Z,reported shipping error,6
2025-07-01T21:50:07.481Z,return canceled,6
2025-07-01T21:50:07.481Z,return picked up,9
2025-07-01T21:50:07.481Z,return processed,8
2025-07-01T21:50:07.481Z,return requested,15


## Key Takeaways

1. **Stream Processing Fundamentals**
   - Structured Streaming provides a DataFrame-based streaming API
   - Supports both batch and streaming processing models
   - Handles data consistency and fault tolerance

2. **Sources and Sinks**
   - Multiple input sources available (Rate, File, Kafka, etc.)
   - Various output sinks for different use cases
   - Memory sink useful for testing and debugging

3. **Data Processing**
   - Supports standard DataFrame operations
   - Windowing and watermarking for time-based processing
   - Aggregations and streaming joins

4. **Monitoring and Management**
   - Built-in query monitoring capabilities
   - Progress tracking and metrics
   - Late data handling strategies


Run the cell below to stop the active streaming queries.

In [0]:
%python
for query in spark.streams.active:
    query.stop()


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
